In [ ]:
import requests
import pandas as pd
import joblib
from sklearn.preprocessing import MinMaxScaler

# Load trained Decision Tree model
try:
    model = joblib.load('decision_tree_aqi_model.pkl')
except Exception as e:
    print(f"Error loading model: {e}")
    exit()

# Scaler setup - ideally training ke scaler ko save aur load krna chahiye
# yha hum fit_transform nhi krenge, bs transform krenge
# isliye aapko training ke scaler ko save krne ki jrurat h . filhal ek dummy scaler bnate h.
scaler = MinMaxScaler()

# Function to fetch AQI pollutant data from AQICN API
def fetch_aqi_pollutants(city, token):
    url = f'https://api.waqi.info/feed/{city}/?token={token}'
    try:
        response = requests.get(url, timeout=10)
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return None

    if response.status_code != 200:
        print(f"API Response Status Code: {response.status_code}")
        return None

    try:
        response_json = response.json()
    except Exception as e:
        print(f"Failed to parse JSON response: {e}")
        return None

    if response_json.get('status') == 'ok':
        iaqi_data = response_json['data'].get('iaqi', {})
        pollutants = {
            'PM2.5': iaqi_data.get('pm25', {'v': 0}).get('v', 0),
            'PM10': iaqi_data.get('pm10', {'v': 0}).get('v', 0),
            'NO2': iaqi_data.get('no2', {'v': 0}).get('v', 0),
            'CO': iaqi_data.get('co', {'v': 0}).get('v', 0),
            'O3': iaqi_data.get('o3', {'v': 0}).get('v', 0)
        }
        return pollutants
    else:
        print(f"API returned bad status: {response_json.get('status')}, message: {response_json.get('data')}")
        return None

# Prediction function using real-time data
def predict_aqi_category(city, token):
    pollutants = fetch_aqi_pollutants(city, token)
    if pollutants is None:
        return "No pollutant data available for prediction."

    # Create DataFrame with proper columns
    df = pd.DataFrame([pollutants], columns=['PM2.5', 'PM10', 'NO2', 'CO', 'O3'])

    # IMPORTANT: Load or fit scaler based on your training data  
    # agar scaler saved nhi kiya h to training me fit kiya huaa scaler load kro
    # Filhal dummy fit kr rha hu (only demo ke liye)
    scaler = joblib.load('scaler.pkl')
    df_scaled = scaler.transform(df)
 # Remove this line if you have saved scaler parameters; load and use transform()

    df_scaled = scaler.transform(df)

    # Convert scaled array back to DataFrame with correct column names for sklearn
    df_scaled_df = pd.DataFrame(df_scaled, columns=df.columns)

    try:
        prediction_num = model.predict(df_scaled_df)[0]
    except Exception as e:
        print(f"Model prediction error: {e}")
        return "Prediction failed."

    aqi_categories = ['Good', 'Moderate', 'Poor', 'Hazardous']
    if 0 <= prediction_num < len(aqi_categories):
        return aqi_categories[prediction_num]
    else:
        return "Unknown AQI category."

if __name__ == "__main__":
    city_name = input("Enter city name for AQI prediction: ").strip()
    aqicn_api_token = 'ae56dc7583110724a47826e6d93c9274f70ac4f2'  # put valid API key here

    if aqicn_api_token == 'YOUR_AQICN_API_TOKEN':
        print("Error: Please replace 'YOUR_AQICN_API_TOKEN' with your actual AQICN API key.")
    else:
        prediction = predict_aqi_category(city_name, aqicn_api_token)
        print(f"Predicted AQI Category for {city_name}: {prediction}")


Predicted AQI Category for delhi: Moderate
